# 🚲 Pandas на реальних даних: EDA прокату велосипедів
## Практичний ноутбук із прикладами й домашніми завданнями

1. прочитайте пояснення 
2. **запустіть готовий приклад** 
3. **виконайте  завдання** 

## 0. Де взяти датасет і як підготувати файл

**Офіційне джерело:** [Bike Sharing, UCI Machine Learning Repository](https://archive.ics.uci.edu/dataset/275/bike+sharing+dataset).  
**Пряме посилання на ZIP:** [Bike Sharing Dataset.zip](https://archive.ics.uci.edu/static/public/275/bike+sharing+dataset.zip).

1. Самостійно завантажте архів за посиланням.
2. Розпакуйте його.
3. Знайдіть **`hour.csv`** — саме його використовуємо. Файл `day.csv` **не** потрібен.
4. Покладіть `hour.csv` **у ту саму папку, що й цей `.ipynb`**. У Google Colab — вручну завантажте `hour.csv` через ліву панель **Files**.
5. Виконайте наступну комірку. Якщо бачите `FileNotFoundError`, перевірте назву файлу та його розташування.

**Чому `hour.csv`?** Кожен рядок — одна година спостереження, а не одна конкретна оренда. Наприклад, `cnt = 250` означає *250 оренд протягом години*. Це важливо, бо **кількість рядків ≠ кількість оренд**.

**Джерело та ліцензія:** Fanaee-T, H. (2013), *Bike Sharing*, UCI, CC BY 4.0, [DOI](https://doi.org/10.24432/C5W894).

In [1]:
# Єдиний імпорт у цьому ноутбуці: тільки Pandas.
import pandas as pd

# Файл hour.csv ви завантажили самостійно й поклали поруч із ноутбуком.
# parse_dates перетворює текстову дату на зручний для аналізу тип datetime.
df = pd.read_csv("hour.csv", parse_dates=["dteday"])

print("Файл прочитано!")
print("Рядків і колонок:", df.shape)
df.head(3)

Файл прочитано!
Рядків і колонок: (17379, 17)


,instant,dteday,season,yr,mnth,hr,holiday,weekday,workingday,weathersit,temp,atemp,hum,windspeed,casual,registered,cnt
0,1,2011-01-01,1,0,1,0,0,6,0,1,0.24,0.2879,0.81,0.0,3,13,16
1,2,2011-01-01,1,0,1,1,0,6,0,1,0.22,0.2727,0.80,0.0,8,32,40
2,3,2011-01-01,1,0,1,2,0,6,0,1,0.22,0.2727,0.80,0.0,5,27,32


## Словничок колонок: що саме ми аналізуємо

| Колонка | Значення | На що звернути увагу |
|---|---|---|
| `instant` | номер запису | це ID, а не кількість оренд |
| `dteday` | календарна дата | у нас є кілька годин на одну дату |
| `hr` | година доби: 0–23 | не плутайте з номером рядка |
| `yr` | `0` = 2011; `1` = 2012 | це категорія, а не «нульовий рік» |
| `mnth` | місяць: 1–12 | ціле число, але календарна категорія |
| `season` | 1 — зима, 2 — весна, 3 — літо, 4 — осінь | числові коди категорій |
| `weekday` | день тижня: 0–6 | 0 = неділя, 6 = субота |
| `holiday` | 1 — свято, 0 — ні | окрема ознака від `workingday` |
| `workingday` | 1 — робочий день, 0 — вихідний або свято | не рівнозначно `holiday` |
| `weathersit` | 1 — ясніше, 2 — хмарно/туманно, 3 — опади, 4 — складна погода | категорії, а не шкала з рівними інтервалами |
| `temp` | нормалізована температура повітря | для `hour.csv` у °C: `temp * 47 - 8` |
| `atemp` | нормалізована відчутна температура | для `hour.csv` у °C: `atemp * 66 - 16` |
| `hum` | відносна вологість, поділена на 100 | у %: `hum * 100` |
| `windspeed` | нормалізована швидкість вітру | не сприймайте як км/год без перетворення |
| `casual` | кількість оренд незареєстрованими клієнтами | число оренд за годину |
| `registered` | кількість оренд зареєстрованими клієнтами | число оренд за годину |
| `cnt` | усі оренди за годину | **`cnt = casual + registered`** |

**Важливі пастки EDA:**

- `cnt`, `casual`, `registered` пов'язані арифметично. Їхня кореляція не є несподіваним відкриттям.
- Середня кількість оренд **за годину** та **загальна кількість оренд за місяць** — різні метрики.
- Цифрові коди `season` і `weathersit` — категорії, тож кореляцію Пірсона з ними треба трактувати обережно.
- Наші висновки описують історичні спостереження 2011–2012 років, а не автоматично поточний ринок.

### Три правила виконання

1. Після кожного прикладу **не просто скопіюйте код**, а змініть колонки, умови або агрегацію за змістом завдання.
2. **Відповідь — це не тільки код.** Напишіть хоча б 1–3 речення з конкретним числом і правильними одиницями вимірювання.
3. Якщо бачите порожню клітинку для завдання з коментарями `TODO`, впишіть власний код **під коментарем**. Комірки з лише коментарями не падають під час запуску.

**Мініприклад правильного висновку (числа умовні):** «У групі A середня кількість оренд становить 150 на годину, а у групі B — 120. Різниця становить 30 оренд за годину; однак це порівняння не доводить, що сама ознака групи є причиною різниці».

---
# Частина 1. Знайомимося з таблицею

Почнемо з базових команд. Зазвичай це перший етап EDA: зрозуміти, які дані взагалі є, що означає один рядок, які типи значень і наскільки вони різноманітні.

## 1. Перегляд рядків: `head()`, `tail()`, `sample()`

**Навіщо це потрібно?** Перш ніж рахувати метрики, варто побачити живі приклади записів: чи схожі вони на те, що ви очікували?

`head(n)` показує перші `n` рядків. `tail(n)` — останні. `sample(n, random_state=42)` обирає випадкові рядки; фіксований `random_state` дозволяє повторити вибір. Перші рядки не завжди показують усе різноманіття датасету.

**Приклад — виконайте комірку та розгляньте результат:**

In [2]:
# Приклад: подивимося на перші 4 погодинні записи.
df.head(4)

,instant,dteday,season,yr,mnth,hr,holiday,weekday,workingday,weathersit,temp,atemp,hum,windspeed,casual,registered,cnt
0,1,2011-01-01,1,0,1,0,0,6,0,1,0.24,0.2879,0.81,0.0,3,13,16
1,2,2011-01-01,1,0,1,1,0,6,0,1,0.22,0.2727,0.80,0.0,8,32,40
2,3,2011-01-01,1,0,1,2,0,6,0,1,0.22,0.2727,0.80,0.0,5,27,32
3,4,2011-01-01,1,0,1,3,0,6,0,1,0.24,0.2879,0.75,0.0,3,10,13


**✍️ Завдання 1 — тепер ви:**

1. Покажіть **останні 6 рядків**.  
2. Виведіть **5 випадкових рядків** так, щоб вибір був повторюваний.  
3. Порівняйте: чи всі приклади стосуються однієї дати?

> **Підказка:** `df.tail(...)` та `df.sample(...)`. Для показу двох таблиць у одній комірці можна написати `print(...)` або створити дві комірки.

In [3]:
print(df.tail(6))
print(df.sample(5, random_state=42))
print(df.sample(5, random_state=42)["dteday"].nunique())

       instant     dteday  season  yr  mnth  hr  holiday  weekday  workingday  \
17373    17374 2012-12-31       1   1    12  18        0        1           1   
17374    17375 2012-12-31       1   1    12  19        0        1           1   
17375    17376 2012-12-31       1   1    12  20        0        1           1   
17376    17377 2012-12-31       1   1    12  21        0        1           1   
17377    17378 2012-12-31       1   1    12  22        0        1           1   
17378    17379 2012-12-31       1   1    12  23        0        1           1   

       weathersit  temp   atemp   hum  windspeed  casual  registered  cnt  
17373           2  0.26  0.2727  0.48     0.1343      10         112  122  
17374           2  0.26  0.2576  0.60     0.1642      11         108  119  
17375           2  0.26  0.2576  0.60     0.1642       8          81   89  
17376           1  0.26  0.2576  0.60     0.1642       7          83   90  
17377           1  0.26  0.2727  0.56     0.1343    

**Напишіть словами:** Чим випадкова вибірка корисна на відміну від перегляду лише початку файлу?

**Відповідь:** Випадкові рядки беруться з різних місць таблиці, тому видно різні дати, сезони й години. Початок файлу показує тільки 1 січня 2011. Останні 6 рядків належать до однієї дати: 31.12.2012. А 5 випадкових рядків стосуються 5 різних дат, тобто не однієї.

## 2. Розмір і назви колонок: `shape`, `columns`, `nunique()`

**Навіщо це потрібно?** Допомагає перевірити, що файл не порожній, знайти ключові поля та зрозуміти кількість унікальних значень.

`shape` — пара `(кількість рядків, кількість колонок)`. `columns` — назви колонок, `nunique()` — кількість різних значень. **Унікальні дати** відрізняються від кількості погодинних рядків.

**Приклад — виконайте комірку та розгляньте результат:**

In [4]:
# Приклад: кількість колонок і унікальних дат.
print("Колонок:", df.shape[1])
print("Унікальних дат:", df["dteday"].nunique())

Колонок: 17
Унікальних дат: 731


**✍️ Завдання 2 — тепер ви:**

1. Виведіть **усі назви колонок**.  
2. Скільки унікальних значень має `hr`? Чи це очікувано?  
3. Скільки унікальних значень має `instant`? Порівняйте з кількістю рядків.

> **Підказка:** Скористайтесь `df.columns`, `df["hr"].nunique()`, `df["instant"].nunique()`.

In [5]:
print(df.columns)
print(df["hr"].nunique())
print(df["instant"].nunique(), len(df))

Index(['instant', 'dteday', 'season', 'yr', 'mnth', 'hr', 'holiday', 'weekday',
       'workingday', 'weathersit', 'temp', 'atemp', 'hum', 'windspeed',
       'casual', 'registered', 'cnt'],
      dtype='str')
24
17379 17379


**Напишіть словами:** Чому кількість унікальних дат набагато менша від кількості рядків?

**Відповідь:** У таблиці 17379 рядків, а унікальних дат лише 731. Це тому, що на одну дату є до 24 рядків, по одному на кожну годину. Унікальних годин `hr` рівно 24, це очікувано, бо в добі 24 години. Унікальних `instant` 17379, стільки ж, скільки рядків, бо це номер запису.

## 3. Типи даних: `info()`, `dtypes`, `select_dtypes()`

**Навіщо це потрібно?** Перед аналізом потрібно знати, де числа, де дати, а де текст. Неправильний тип може зламати обчислення або спотворити сенс.

`info()` показує тип, кількість непорожніх значень та пам’ять. `dtypes` повертає типи колонок. Pandas може зберігати **категорії як цілі числа** — це не робить їх безперервними величинами.

**Приклад — виконайте комірку та розгляньте результат:**

In [6]:
# Приклад: переглядаємо структуру таблиці.
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 17379 entries, 0 to 17378
Data columns (total 17 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   instant     17379 non-null  int64         
 1   dteday      17379 non-null  datetime64[us]
 2   season      17379 non-null  int64         
 3   yr          17379 non-null  int64         
 4   mnth        17379 non-null  int64         
 5   hr          17379 non-null  int64         
 6   holiday     17379 non-null  int64         
 7   weekday     17379 non-null  int64         
 8   workingday  17379 non-null  int64         
 9   weathersit  17379 non-null  int64         
 10  temp        17379 non-null  float64       
 11  atemp       17379 non-null  float64       
 12  hum         17379 non-null  float64       
 13  windspeed   17379 non-null  float64       
 14  casual      17379 non-null  int64         
 15  registered  17379 non-null  int64         
 16  cnt         17379 non-null  int64

**✍️ Завдання 3 — тепер ви:**

1. Виведіть типи колонок `dteday`, `hr`, `cnt`, `temp`.  
2. Отримайте список **усіх числових колонок**.  
3. Назвіть **три колонки**, які технічно числові, але за змістом є категоріями.

> **Підказка:** Тип конкретної колонки — `df["cnt"].dtype`; вибір числових — `df.select_dtypes(include="number").columns`.

In [7]:
print(df[["dteday", "hr", "cnt", "temp"]].dtypes)
print(df.select_dtypes(include="number").columns)

dteday    datetime64[us]
hr                 int64
cnt                int64
temp             float64
dtype: object
Index(['instant', 'season', 'yr', 'mnth', 'hr', 'holiday', 'weekday',
       'workingday', 'weathersit', 'temp', 'atemp', 'hum', 'windspeed',
       'casual', 'registered', 'cnt'],
      dtype='str')


**Напишіть словами:** Чому не можна трактувати середнє значення `season` як «середню пору року»?

**Відповідь:** `season` це не число, а код сезону: 1, 2, 3, 4. Середнє 2.5 нічого не означає, бо «пів сезону» не існує. Три колонки, які числові, але за змістом є категоріями: `season`, `weathersit`, `weekday`.

## 4. Описові статистики: `describe()`, `mean()`, `median()`

**Навіщо це потрібно?** Середнє, медіана та крайні значення допомагають побачити типовий попит і нерівномірність розподілу.

`mean()` — арифметичне середнє, `median()` — число посередині відсортованого набору. `describe()` показує кількість, середнє, стандартне відхилення, мінімум, квартилі та максимум. Якщо середнє значно вище медіани, варто перевірити наявність великих значень.

**Приклад — виконайте комірку та розгляньте результат:**

In [8]:
# Приклад: статистики загального числа оренд за годину.
df["cnt"].describe()

count    17379.000000
mean       189.463088
std        181.387599
min          1.000000
25%         40.000000
50%        142.000000
75%        281.000000
max        977.000000
Name: cnt, dtype: float64

**✍️ Завдання 4 — тепер ви:**

1. Обчисліть **середню та медіанну** кількість оренд `casual` за годину.  
2. Знайдіть **мінімальне й максимальне** значення `registered`.  
3. Яка із двох груп — `casual` чи `registered` — має більший розкид значень? Підтвердьте числом.

> **Підказка:** `df["casual"].agg(["mean", "median", "std"])` поверне одразу декілька статистик.

In [9]:
print(df["casual"].agg(["mean", "median"]))
print(df["registered"].agg(["min", "max"]))
print(df[["casual", "registered"]].std())
print(df[["casual", "registered"]].describe())

mean      35.676218
median    17.000000
Name: casual, dtype: float64
min      0
max    886
Name: registered, dtype: int64
casual         49.305030
registered    151.357286
dtype: float64
             casual    registered
count  17379.000000  17379.000000
mean      35.676218    153.786869
std       49.305030    151.357286
min        0.000000      0.000000
25%        4.000000     34.000000
50%       17.000000    115.000000
75%       48.000000    220.000000
max      367.000000    886.000000


**Напишіть словами:** Яка метрика краще описує «типову» годину для одного з типів клієнтів і чому?

**Відповідь:** Для `casual` краще медіана. Вона дорівнює 17 оренд за годину, а середнє 35.7, бо його підіймають рідкі години з дуже великими значеннями (максимум 367). Для `registered` мінімум 0, максимум 886. Стандартне відхилення у `casual` 49.3, у `registered` 151.4, тож у `registered` числа розкидані більше, але відносно свого середнього (35.7 і 153.8) розкид більший у `casual`.

## 5. Частоти категорій: `value_counts()`

**Навіщо це потрібно?** Так можна побачити, які категорії зустрічаються часто, а яких майже немає. Малочисельні групи варто аналізувати обережно.

`value_counts()` рахує частоти. `normalize=True` перетворює їх на частки від 0 до 1. Частка 0.25 означає 25% *рядків*, а не обов’язково 25% *оренд*.

**Приклад — виконайте комірку та розгляньте результат:**

In [10]:
# Приклад: скільки записів належить до кожного сезону.
print(df["season"].value_counts().sort_index())
print("Частки записів:")
print((df["season"].value_counts(normalize=True).sort_index() * 100).round(1))

season
1    4242
2    4409
3    4496
4    4232
Name: count, dtype: int64
Частки записів:
season
1    24.4
2    25.4
3    25.9
4    24.4
Name: proportion, dtype: float64


**✍️ Завдання 5 — тепер ви:**

1. Порахуйте, скільки годинних записів припадає на **кожну категорію погоди** (`weathersit`).  
2. Переведіть ці частки у відсотки.  
3. Якої категорії найменше? Чому її середній попит може бути ненадійним для висновків?

> **Підказка:** Спробуйте `df["weathersit"].value_counts(normalize=True) * 100`.

In [11]:
print(df["weathersit"].value_counts().sort_index())
print((df["weathersit"].value_counts(normalize=True).sort_index() * 100).round(2))

weathersit
1    11413
2     4544
3     1419
4        3
Name: count, dtype: int64
weathersit
1    65.67
2    26.15
3     8.17
4     0.02
Name: proportion, dtype: float64


**Напишіть словами:** Скільки спостережень є в найрідкіснішій погодній категорії?

**Відповідь:** У найрідкіснішій категорії погоди (код 4) є лише 3 спостереження, це 0.02% усіх рядків. Для порівняння: код 1 має 11413 рядків (65.67%), код 2 має 4544 (26.15%), код 3 має 1419 (8.17%). Середнє за 3 години ненадійне, бо одна година може сильно змінити результат.

---
# Частина 2. Відбір, сортування та якість

У реальних задачах аналітик постійно ставить уточнювальні питання до таблиці: «покажи лише вихідні», «знайди пікові години», «чи точно всі рядки коректні?».

## 6. Вибір частини даних: колонки та `loc`

**Навіщо це потрібно?** Часто для питання потрібні не всі 17 колонок, а лише кілька.

Список у подвійних квадратних дужках обирає **кілька колонок**. `loc[умова, список_колонок]` одночасно відбирає **рядки за умовою** та **потрібні колонки**. `iloc` обирає рядки/стовпці за позиціями, а не за назвами.

**Приклад — виконайте комірку та розгляньте результат:**

In [12]:
# Приклад: дата, година і кількість оренд для перших п'яти рядків.
df.loc[:, ["dteday", "hr", "cnt"]].head(5)

,dteday,hr,cnt
0,2011-01-01,0,16
1,2011-01-01,1,40
2,2011-01-01,2,32
3,2011-01-01,3,13
4,2011-01-01,4,1


**✍️ Завдання 6 — тепер ви:**

1. Покажіть **лише** `dteday`, `hr`, `temp`, `weathersit`, `cnt` для перших 10 рядків.  
2. Через `iloc` покажіть перші 3 рядки та перші 4 колонки.  
3. Поясніть різницю між вибором за назвою і за позицією.

> **Підказка:** `df.loc[:, [...]]` для назв, `df.iloc[:3, :4]` для позицій.

In [13]:
print(df.loc[:, ["dteday", "hr", "temp", "weathersit", "cnt"]].head(10))
print(df.iloc[:3, :4])

      dteday  hr  temp  weathersit  cnt
0 2011-01-01   0  0.24           1   16
1 2011-01-01   1  0.22           1   40
2 2011-01-01   2  0.22           1   32
3 2011-01-01   3  0.24           1   13
4 2011-01-01   4  0.24           1    1
5 2011-01-01   5  0.24           2    1
6 2011-01-01   6  0.22           1    2
7 2011-01-01   7  0.20           1    3
8 2011-01-01   8  0.24           1    8
9 2011-01-01   9  0.32           1   14
   instant     dteday  season  yr
0        1 2011-01-01       1   0
1        2 2011-01-01       1   0
2        3 2011-01-01       1   0


**Напишіть словами:** Коли відбір колонок робить результат зручнішим для читання?

**Відповідь:** Коли колонок багато, а потрібно лише кілька, відбір робить таблицю коротшою, і її легше читати. Різниця між способами така: `loc` бере колонки за назвою, а `iloc` бере за номером позиції. Якщо колонки поміняти місцями, то `iloc` покаже вже інші дані.

## 7. Фільтрація: просте порівняння

**Навіщо це потрібно?** Майже всі бізнес-запитання стосуються підмножин даних: робочі дні, певний рік, дощові години.

Умова `df["workingday"] == 1` утворює маску з `True/False` для кожного рядка. Якщо передати її в `df[...]`, залишаться лише рядки з `True`. Після фільтра можна обчислювати метрики.

**Приклад — виконайте комірку та розгляньте результат:**

In [14]:
# Приклад: лише робочі дні.
work = df[df["workingday"] == 1]
print("Погодинних записів у робочі дні:", len(work))
print("Середні оренди за годину:", round(work["cnt"].mean(), 1))

Погодинних записів у робочі дні: 11865
Середні оренди за годину: 193.2


**✍️ Завдання 7 — тепер ви:**

1. Відберіть **лише святкові дні** (`holiday == 1`).  
2. Порахуйте, скільки погодинних записів відповідають цій умові.  
3. Знайдіть медіану `cnt` для святкових днів.

> **Підказка:** Створіть нову змінну на кшталт `holidays = df[df["holiday"] == 1]`.

In [15]:
holidays = df[df["holiday"] == 1]
print(len(holidays))
print(holidays["cnt"].median())
print(holidays["dteday"].nunique())

500
97.0
21


**Напишіть словами:** Чому кількість рядків зі святами не дорівнює кількості святкових дат?

**Відповідь:** Одна святкова дата має до 24 рядків, по одному на кожну годину. Тому рядків 500, а святкових дат лише 21. Медіана `cnt` у святкові дні дорівнює 97 орендам за годину.

## 8. Складні умови: `&`, `|`, `between()`, `isin()`

**Навіщо це потрібно?** Окрема умова рідко відповідає на справжнє питання. Частіше цікавить поєднання години, погоди й типу дня.

`&` означає **і**, `|` — **або**. Кожну умову ставте в круглі дужки. `between(7, 9)` включає межі 7 та 9. `isin([...])` перевіряє входження у список.

**Приклад — виконайте комірку та розгляньте результат:**

In [16]:
# Приклад: ранкові години 7–9 у робочі дні.
morning_work = df[(df["hr"].between(7, 9)) & (df["workingday"] == 1)]
print("Рядків:", len(morning_work))
print("Середній cnt:", morning_work["cnt"].mean())

Рядків: 1488
Середній cnt: 336.3790322580645


**✍️ Завдання 8 — тепер ви:**

1. Виберіть **неробочі дні**, години **17–20 включно**.  
2. Порахуйте середнє `cnt` у цій групі.  
3. Окремо виберіть усі записи, де погода має код **2 або 3**. Скільки їх?

> **Підказка:** Спробуйте `df[(...) & (...)]` та `df["weathersit"].isin([2, 3])`.

In [17]:
weekend_evening = df[(df["workingday"] == 0) & (df["hr"].between(17, 20))]
print(len(weekend_evening))
print(weekend_evening["cnt"].mean())
bad_weather = df[df["weathersit"].isin([2, 3])]
print(len(bad_weather))

921
252.83170466883823
5963


**Напишіть словами:** Як змінюється сенс порівняння, якщо одночасно враховувати і тип дня, і час?

**Відповідь:** Коли беремо разом тип дня й час, порівняння стає точнішим. У неробочі дні з 17 до 20 години 921 рядок, і середній `cnt` там 252.8 оренди за годину. Записів із погодою 2 або 3 є 5963. Якщо дивитися на весь день разом, ці різниці губляться.

## 9. Сортування і пошук крайніх значень

**Навіщо це потрібно?** Так можна швидко знайти години рекордного попиту, найбільш холодні спостереження чи підозрілі значення.

`sort_values("назва")` впорядковує рядки. `ascending=False` — від найбільшого до найменшого. `nlargest(n, "колонка")` швидко повертає `n` найбільших значень. Важливо дивитися **інші колонки цих рядків**, щоб зрозуміти контекст.

**Приклад — виконайте комірку та розгляньте результат:**

In [18]:
# Приклад: 5 найтепліших спостережень (за нормалізованою temp).
df.nlargest(5, "temp")[["dteday", "hr", "temp", "cnt"]]

,dteday,hr,temp,cnt
13163,2012-07-07,16,1.00,294
12972,2012-06-29,17,0.98,539
4768,2011-07-22,14,0.96,101
4934,2011-07-29,12,0.96,135
4936,2011-07-29,14,0.96,148


**✍️ Завдання 9 — тепер ви:**

1. Знайдіть **10 годин із найбільшою кількістю оренд**.  
2. Виведіть для них дату, годину, погоду та `cnt`.  
3. Знайдіть **5 годин із найменшою кількістю оренд**. Як ви пояснили б ці значення?

> **Підказка:** `df.nlargest(10, "cnt")[[...]]` і `df.nsmallest(5, "cnt")`.

In [19]:
print(df.nlargest(10, "cnt")[["dteday", "hr", "weathersit", "cnt"]])
print(df.nsmallest(5, "cnt")[["dteday", "hr", "weathersit", "cnt"]])

          dteday  hr  weathersit  cnt
14773 2012-09-12  18           1  977
14964 2012-09-20  17           1  976
14748 2012-09-11  17           1  970
14725 2012-09-10  18           1  968
15084 2012-09-25  17           1  967
15780 2012-10-24  17           1  963
10622 2012-03-23  17           2  957
15108 2012-09-26  17           1  953
15444 2012-10-10  17           1  948
15588 2012-10-16  17           1  943
       dteday  hr  weathersit  cnt
4  2011-01-01   4           1    1
5  2011-01-01   5           2    1
30 2011-01-02   7           2    1
49 2011-01-03   4           1    1
71 2011-01-04   2           1    1


**Напишіть словами:** Чому рекордні години не завжди означають помилки у даних?

**Відповідь:** У топ-10 години мають від 943 до 977 оренд. 8 з 10 припадають на 17:00, решта на 18:00, а майже всі це вересень або жовтень 2012 року, при ясній погоді. Це звичайний вечірній пік у гарну погоду, а не помилка. Найменші 5 годин мають по 1 оренді, це майже все глибока ніч (2–5 година) і один раз 7 ранку, коли їде мало людей.

## 10. Пропуски, дублікати та перевірка логіки

**Навіщо це потрібно?** EDA — це не лише пошук закономірностей. Спочатку потрібно переконатись, що базові співвідношення в таблиці не порушені.

`isna().sum()` рахує порожні значення за колонками. `duplicated().sum()` рахує повторення **повних рядків**, але два рядки за ту саму дату — не обов’язково дублікати, адже години різні. Якість даних включає також перевірки бізнес-правил.

**Приклад — виконайте комірку та розгляньте результат:**

In [20]:
# Приклад: кількість пропущених значень за колонками.
missing = df.isna().sum().sort_values(ascending=False)
print(missing.head(10))

instant       0
dteday        0
season        0
yr            0
mnth          0
hr            0
holiday       0
weekday       0
workingday    0
weathersit    0
dtype: int64


**✍️ Завдання 10 — тепер ви:**

1. Порахуйте **повні дублікати**.  
2. Перевірте, чи є повторні поєднання **дата + година**.  
3. Порахуйте рядки, де **`cnt != casual + registered`**.  
4. Знайдіть, чи є `cnt` меншим за нуль.

> **Підказка:** Для правила суми: `(df["cnt"] != df["casual"] + df["registered"]).sum()`.

In [21]:
print(df.duplicated().sum())
print(df.duplicated(subset=["dteday", "hr"]).sum())
print((df["cnt"] != df["casual"] + df["registered"]).sum())
print((df["cnt"] < 0).sum())

0
0
0
0


**Напишіть словами:** Навіть якщо проблем немає, чому важливо було їх перевірити?

**Відповідь:** Перевірка показала, що пропусків немає, повних дублікатів 0, повторів дата + година 0, порушень формули `cnt = casual + registered` 0 і від'ємних оренд 0. Але без перевірки ми б цього не знали. Також видно, що бракує 165 годин: 731 день × 24 години = 17544, а рядків 17379.

## 11. Нові колонки: перетворюємо шкали

**Навіщо це потрібно?** Нормалізовані числа 0–1 складніше інтерпретувати. У EDA часто варто додати зрозуміліші одиниці вимірювання.

Створення `df["нова"] = ...` додає колонку. У вихідному `hour.csv` `temp` нормалізовано за шкалою від −8 до +39 °C, а `hum` — поділено на 100. Виконуйте перетворення в **нових колонках**, а не перезаписуйте оригінальні.

**Приклад — виконайте комірку та розгляньте результат:**

In [22]:
# Приклад: температура у градусах Цельсія.
df["temp_c"] = df["temp"] * 47 - 8
df[["temp", "temp_c"]].head()

,temp,temp_c
0,0.24,3.28
1,0.22,2.34
2,0.22,2.34
3,0.24,3.28
4,0.24,3.28


**✍️ Завдання 11 — тепер ви:**

1. Створіть колонку `humidity_pct` — вологість у **відсотках**.  
2. Створіть `feels_like_c` — відчутну температуру у **°C**.  
3. Покажіть 5 рядків із `hum`, `humidity_pct`, `atemp`, `feels_like_c`.

> **Підказка:** Формули є у словнику колонок; значення для відчутної температури: діапазон від −16 до +50 °C.

In [23]:
df["humidity_pct"] = df["hum"] * 100
df["feels_like_c"] = df["atemp"] * 66 - 16
print(df[["hum", "humidity_pct", "atemp", "feels_like_c"]].head())

    hum  humidity_pct   atemp  feels_like_c
0  0.81          81.0  0.2879        3.0014
1  0.80          80.0  0.2727        1.9982
2  0.80          80.0  0.2727        1.9982
3  0.75          75.0  0.2879        3.0014
4  0.75          75.0  0.2879        3.0014


**Напишіть словами:** Навіщо в бізнес-звіті показувати 20 °C замість нормалізованого значення?

**Відповідь:** Людина розуміє 20 °C, а значення 0.5 їй нічого не говорить. У бізнес-звіті треба писати зрозумілі одиниці, наприклад 81% вологості замість 0.81.

---
# Частина 3. Шукаємо закономірності

Тепер ми вже вміємо брати потрібні записи й перевіряти якість. Далі — найкорисніша частина Pandas для аналітики: порівнювати групи та перетин двох ознак.

## 12. Просте групування: `groupby()`

**Навіщо це потрібно?** Питання «в яку пору року попит вищий?» не вирішити простим `mean()` по всій таблиці — треба рахувати середнє **окремо для груп**.

`groupby("season")["cnt"].mean()` формує одну середню величину **на сезон**. `mean()` — середня кількість оренд *за спостережену годину*, а `sum()` — загальна кількість оренд у всіх спостережених годинах. Вибір метрики принциповий.

**Приклад — виконайте комірку та розгляньте результат:**

In [24]:
# Приклад: середні оренди за годину в кожному сезоні.
seasonal_mean = df.groupby("season")["cnt"].mean().round(1)
seasonal_mean

season
1    111.1
2    208.3
3    236.0
4    198.9
Name: cnt, dtype: float64

**✍️ Завдання 12 — тепер ви:**

1. Розрахуйте середню кількість оренд **за кожної погодної категорії**.  
2. Відсортуйте результат від більшого до меншого.  
3. Зробіть аналогічне порівняння **за роками** (`yr`).

> **Підказка:** Після групування можна додати `.sort_values(ascending=False).round(1)`.

In [25]:
weather_mean = df.groupby("weathersit")["cnt"].mean().round(1)
print(weather_mean)
print(weather_mean.sort_values(ascending=False))
print(df.groupby("yr")["cnt"].mean().round(1))

weathersit
1    204.9
2    175.2
3    111.6
4     74.3
Name: cnt, dtype: float64
weathersit
1    204.9
2    175.2
3    111.6
4     74.3
Name: cnt, dtype: float64
yr
0    143.8
1    234.7
Name: cnt, dtype: float64


**Напишіть словами:** Де середній попит вищий? Чому таке порівняння ще не доводить причинного впливу погоди?

**Відповідь:** Найвищий середній попит у ясну погоду (код 1): 204.9 оренди за годину. Далі код 2 — 175.2, код 3 — 111.6, код 4 — 74.3 (але це лише 3 години). За роками 2011 дає 143.8, а 2012 дає 234.7, тобто більше на 90.9 оренди за годину. Це не доводить, що погода є причиною, бо дощі частіше бувають в інші сезони й години, а в 2012 році попит і так ріс.

## 13. Кілька статистик одразу: `agg()`

**Навіщо це потрібно?** Одна середня величина приховує кількість спостережень і нерівномірність. Корисно дивитись на середнє, медіану, мінімум і кількість.

`agg()` дозволяє називати вихідні показники. `size` рахує **рядки**, `mean` — середнє, `median` — медіану, `sum` — суму. Не плутайте `size` і кількість оренд.

**Приклад — виконайте комірку та розгляньте результат:**

In [26]:
# Приклад: порівнюємо робочі й неробочі дні.
df.groupby("workingday").agg(
    hours=("cnt", "size"),
    avg_per_hour=("cnt", "mean"),
    median_per_hour=("cnt", "median"),
    total_rentals=("cnt", "sum"),
).round(1)

,hours,avg_per_hour,median_per_hour,total_rentals
workingday,,,,
0,5514,181.4,119.0,1000269
1,11865,193.2,151.0,2292410


**✍️ Завдання 13 — тепер ви:**

1. Побудуйте таблицю **за годинами `hr`**, де є: кількість спостережень, середній `cnt`, медіанний `cnt`, максимальний `cnt`.  
2. Визначте три години з найвищим **середнім** попитом.  
3. Поясніть, чому рейтинг за середнім може відрізнятися від рейтингу за сумою.

> **Підказка:** Візьміть назви вихідних показників за зразком та додайте `max_per_hour=("cnt", "max")`.

In [27]:
by_hour = df.groupby("hr").agg(
    n=("cnt", "size"),
    avg_per_hour=("cnt", "mean"),
    median_per_hour=("cnt", "median"),
    max_per_hour=("cnt", "max"),
    total_rentals=("cnt", "sum"),
).round(1)
print(by_hour)
print(by_hour.sort_values("avg_per_hour", ascending=False).head(3))
print(by_hour.sort_values("total_rentals", ascending=False).head(3))

      n  avg_per_hour  median_per_hour  max_per_hour  total_rentals
hr                                                                 
0   726          53.9             40.0           283          39130
1   724          33.4             20.0           168          24164
2   715          22.9             11.0           132          16352
3   697          11.7              6.0            79           8174
4   697           6.4              6.0            28           4428
5   717          19.9             19.0            66          14261
6   725          76.0             76.0           213          55132
7   727         212.1            208.0           596         154171
8   727         359.0            385.0           839         261001
9   727         219.3            216.0           426         159438
10  727         173.7            147.0           539         126257
11  727         208.1            180.0           663         151320
12  728         253.3            229.0          

**Напишіть словами:** Які години найважливіші для планування кількості доступних велосипедів?

**Відповідь:** Найважливіші години: 17:00 (461.5 оренди), 18:00 (425.5) і 8:00 (359.0). За сумою оренд ці три години теж на перших місцях. Але рейтинг за сумою може відрізнятися від рейтингу за середнім, бо сума залежить ще й від кількості записів у групі, а середнє не залежить. Для планування велосипедів найважливіші ранок о 8 і вечір 17–18.

## 14. Дві ознаки одночасно: `groupby([...])`

**Навіщо це потрібно?** Групування тільки за одним фактором може маскувати різницю між групами. Наприклад, одна й та сама година поводиться по-різному у робочий та неробочий день.

Якщо передати до `groupby` список колонок, Pandas створить групи для **кожної комбінації**. `reset_index()` повертає колонки групування зі спеціального індексу в звичайні колонки.

**Приклад — виконайте комірку та розгляньте результат:**

In [28]:
# Приклад: середні оренди за годинами і типом дня.
by_hour_day = (
    df.groupby(["workingday", "hr"])["cnt"]
      .mean()
      .round(1)
      .reset_index(name="avg_rentals")
)
by_hour_day.head(8)

,workingday,hr,avg_rentals
0,0,0,90.8
1,0,1,69.5
2,0,2,53.2
3,0,3,25.8
4,0,4,8.3
5,0,5,8.7
6,0,6,18.7
7,0,7,43.4


**✍️ Завдання 14 — тепер ви:**

1. Порахуйте середнє `cnt` для **кожної пари сезон + погода**.  
2. Додайте кількість рядків у групі (через `agg`).  
3. Знайдіть комбінації з високим середнім попитом і перевірте, чи достатньо там спостережень.

> **Підказка:** `groupby(["season", "weathersit"]).agg(avg=("cnt", "mean"), n=("cnt", "size"))`.

In [29]:
season_weather = df.groupby(["season", "weathersit"]).agg(
    avg=("cnt", "mean"),
    n=("cnt", "size"),
).round(1).reset_index()
print(season_weather.sort_values("avg", ascending=False))

    season  weathersit    avg     n
7        3           1  244.5  3280
4        2           1  232.4  2859
8        3           2  224.8   947
10       4           1  209.5  2609
11       4           2  201.5  1248
5        2           2  181.3  1144
9        3           3  172.0   269
0        1           1  122.1  2665
12       4           3  116.5   375
6        2           3  115.2   406
1        1           2  103.1  1205
3        1           4   74.3     3
2        1           3   58.7   369


**Напишіть словами:** Чому середнє, обчислене за 2 годинами, менш стабільне, ніж за 200 годинами?

**Відповідь:** У 2 годинах одне випадкове значення сильно змінює середнє, а у 200 годинах окремі дивні значення губляться серед інших. Найвищі середні: літо + ясно 244.5 (3280 годин), весна + ясно 232.4 (2859 годин), літо + хмарно 224.8 (947 годин), тому тут спостережень достатньо. Комбінація зима + складна погода має лише 3 години (середнє 74.3), їй довіряти не можна.

## 15. Таблиця «рядки × стовпці»: `pivot_table()`

**Навіщо це потрібно?** Зі списку груп не завжди легко порівнювати значення. Pivot-таблиця створює компактну матрицю: по рядках один фактор, по колонках — інший.

`pivot_table(index=..., columns=..., values=..., aggfunc="mean")` показує середнє для кожного перетину. Це не новий спосіб обчислення середнього, а **зручна форма** групування.

**Приклад — виконайте комірку та розгляньте результат:**

In [30]:
# Приклад: години × тип дня, середня кількість оренд.
df.pivot_table(
    index="hr", columns="workingday", values="cnt", aggfunc="mean"
).round(1)

workingday,0,1
hr,,
0,90.8,36.8
1,69.5,16.6
2,53.2,8.7
3,25.8,4.9
4,8.3,5.4
5,8.7,24.9
6,18.7,102.5
7,43.4,290.6
8,105.7,477.0


**✍️ Завдання 15 — тепер ви:**

1. Створіть таблицю **місяць × рік**, у якій записане середнє `cnt` за годину.  
2. Для яких місяців різниця між 2011 та 2012 роками найбільша?  
3. В окремій таблиці порівняйте **сезон × тип дня**.

> **Підказка:** Після побудови `table` відніміть `table[0]` від `table[1]`, якщо обидві колонки присутні.

In [31]:
table = df.pivot_table(index="mnth", columns="yr", values="cnt", aggfunc="mean").round(1)
print(table)
diff = (table[1] - table[0]).round(1)
print(diff.sort_values(ascending=False))
print(df.pivot_table(index="season", columns="workingday", values="cnt", aggfunc="mean").round(1))

yr        0      1
mnth              
1      55.5  130.6
2      74.3  149.0
3      87.7  221.9
4     131.9  242.7
5     182.6  263.3
6     199.3  281.7
7     190.0  273.7
8     187.0  288.3
9     177.7  303.6
10    166.2  280.8
11    142.1  212.6
12    117.8  166.7


mnth
3     134.2
9     125.9
10    114.6
4     110.8
8     101.3
7      83.7
6      82.4
5      80.7
1      75.1
2      74.7
11     70.5
12     48.9
dtype: float64
workingday      0      1
season                  
1            95.1  119.4
2           214.4  205.7
3           230.3  238.5
4           191.9  202.2


**Напишіть словами:** Чому порівнювати ті самі місяці двох років справедливіше, ніж довільні місяці?

**Відповідь:** Однакові місяці мають однакову пору року й схожу погоду, тому різниця між роками показує саме зміну між роками. Найбільша різниця була в березні (+134.2 оренди за годину), вересні (+125.9) і жовтні (+114.6), найменша в грудні (+48.9). У таблиці сезон × тип дня взимку робочі дні кращі (119.4 проти 95.1), навесні вихідні кращі (214.4 проти 205.7), влітку робочі (238.5 проти 230.3), восени робочі (202.2 проти 191.9).

## 16. Частоти перетинів: `crosstab()`

**Навіщо це потрібно?** Іноді нас цікавлять не середні оренди, а **розподіл спостережень**: скільки годин було ясних чи дощових у різні сезони.

`pd.crosstab()` рахує комбінації категорій. `normalize="index"` ділить кожну клітинку на суму її **рядка**. Тому відсотки в кожному рядку дорівнюють 100%. Це інша постановка питання, ніж середній `cnt`.

**Приклад — виконайте комірку та розгляньте результат:**

In [32]:
# Приклад: частка погодних категорій у кожному сезоні.
pd.crosstab(
    df["season"], df["weathersit"], normalize="index"
).mul(100).round(1)

weathersit,1,2,3,4
season,,,,
1,62.8,28.4,8.7,0.1
2,64.8,25.9,9.2,0.0
3,73.0,21.1,6.0,0.0
4,61.6,29.5,8.9,0.0


**✍️ Завдання 16 — тепер ви:**

1. Створіть **таблицю кількості** спостережень для `workingday × weathersit`.  
2. Створіть **таблицю відсотків у межах кожного типу дня**.  
3. Поясніть, що означає одна обрана вами клітинка (який знаменник?).

> **Підказка:** `pd.crosstab(df["workingday"], df["weathersit"], normalize="index")`.

In [33]:
print(pd.crosstab(df["workingday"], df["weathersit"]))
print(pd.crosstab(df["workingday"], df["weathersit"], normalize="index").mul(100).round(1))

weathersit     1     2     3  4
workingday                     
0           3781  1353   379  1
1           7632  3191  1040  2


weathersit     1     2    3    4
workingday                      
0           68.6  24.5  6.9  0.0
1           64.3  26.9  8.8  0.0


**Напишіть словами:** Чим «30% годин були дощовими» відрізняється від «за дощу було 30% усіх оренд»?

**Відповідь:** «30% годин були дощовими» рахує години, де знаменник це всі години. «За дощу було 30% усіх оренд» рахує оренди, де знаменник це всі оренди. Це різні речі, бо в дощ їздять менше. Приклад клітинки: робочі дні, погода 3 дає 8.8%. Це значить, що 8.8% усіх робочих годин (1040 з 11865) були з опадами. Знаменник тут усі робочі години.

## 17. Два типи клієнтів: працюємо з кількома колонками

**Навіщо це потрібно?** Компанії потрібно розуміти не лише весь попит, а й того, хто цими велосипедами користується.

`sum()` за колонкою рахує всі оренди певного типу. Коли сума береться для кількох колонок, результат — окреме число для кожної з них. Під час порівняння годин краще застосовувати середнє **за годину**, щоб різна кількість записів не вводила в оману.

**Приклад — виконайте комірку та розгляньте результат:**

In [34]:
# Приклад: загальні кількості оренд за весь період.
segment_totals = df[["casual", "registered"]].sum()
print(segment_totals)
print("Частка casual, %:", round(segment_totals["casual"] / segment_totals.sum() * 100, 1))

casual         620017
registered    2672662
dtype: int64
Частка casual, %: 18.8


**✍️ Завдання 17 — тепер ви:**

1. Порахуйте середні `casual` і `registered` **за кожну годину доби** (дві колонки в таблиці).  
2. Знайдіть, о котрій годині в кожного сегмента найбільший середній попит.  
3. Повторіть аналіз, розділивши на робочі та неробочі дні.

> **Підказка:** Після `.groupby("hr")[["casual", "registered"]].mean()` можна викликати `.idxmax()`.

In [35]:
by_hour_seg = df.groupby("hr")[["casual", "registered"]].mean().round(1)
print(by_hour_seg)
print(by_hour_seg.idxmax())
by_day_hour_seg = df.groupby(["workingday", "hr"])[["casual", "registered"]].mean().round(1)
print(by_day_hour_seg.groupby(level=0).idxmax())
print(by_day_hour_seg.groupby(level=0).max())

    casual  registered
hr                    
0     10.2        43.7
1      6.5        26.9
2      4.8        18.1
3      2.7         9.0
4      1.3         5.1
5      1.4        18.5
6      4.2        71.9
7     11.1       201.0
8     21.7       337.3
9     30.9       188.4
10    46.5       127.2
11    59.5       148.6
12    68.3       185.0
13    72.3       181.4
14    75.6       165.4
15    74.9       176.3
16    73.7       238.2
17    74.3       387.2
18    61.1       364.4
19    48.8       262.8
20    36.2       189.8
21    28.3       144.1
22    22.3       109.1
23    15.2        72.6
casual        14
registered    17
dtype: int64
             casual registered
workingday                    
0           (0, 14)    (0, 12)
1           (1, 17)    (1, 17)
            casual  registered
workingday                    
0            139.6       240.0
1             56.9       468.4


**Напишіть словами:** Чим можуть пояснюватися відмінності між зареєстрованими та незареєстрованими? Це гіпотеза, а не доведена причина.

**Відповідь:** У `casual` найбільший середній попит о 14:00 (75.6 оренди за годину), а у `registered` о 17:00 (387.2). У робочі дні обидва піки о 17:00 (56.9 і 468.4). У неробочі `casual` має пік о 14:00 (139.6), а `registered` о 12:00 (240.0). Гіпотеза: зареєстровані їздять на роботу чи навчання (піки о 8 і 17), а незареєстровані їздять на прогулянки в середині дня. Усього оренд `casual` 620017 (18.8%), `registered` 2672662.

---
# Частина 4. Складніша аналітика без графіків

Ці прийоми допомагають перейти від очевидних підрахунків до перевірки гіпотез: побачити зв’язок, незвичні значення й поведінку в часі.

## 18. Кореляції: `corr()` та їх обмеження

**Навіщо це потрібно?** Кореляція дозволяє кількісно оцінити лінійний зв’язок між двома числовими величинами.

Кореляція Пірсона близька до +1 за сильного позитивного лінійного зв’язку, близька до −1 за негативного і близька до 0 за слабкого **лінійного** зв’язку. Це **не** доказ причинності. Також не варто бездумно включати ID, кодовані категорії та `casual`/`registered` при аналізі `cnt`.

**Приклад — виконайте комірку та розгляньте результат:**

In [36]:
# Приклад: як кілька погодних показників пов'язані з cnt.
(df[["temp_c", "hum", "windspeed", "cnt"]]
    .corr()["cnt"]
    .sort_values(ascending=False)
    .round(3))

cnt          1.000
temp_c       0.405
windspeed    0.093
hum         -0.323
Name: cnt, dtype: float64

**✍️ Завдання 18 — тепер ви:**

1. Порахуйте кореляцію `cnt` окремо з **`temp_c` і `humidity_pct`**.  
2. Порахуйте кореляцію між `temp_c` і `feels_like_c`.  
3. Поясніть, чому `cnt` та `registered` не варто називати незалежним відкриттям.

> **Підказка:** Використовуйте створені в завданні 11 колонки та `df[[...]].corr()`.

In [37]:
print(df[["temp_c", "humidity_pct", "cnt"]].corr()["cnt"].round(3))
print(df[["temp_c", "feels_like_c"]].corr().round(3))
print((df["cnt"] == df["casual"] + df["registered"]).all())
print(df[["registered", "cnt"]].corr().round(3))

temp_c          0.405
humidity_pct   -0.323
cnt             1.000
Name: cnt, dtype: float64
              temp_c  feels_like_c
temp_c         1.000         0.988
feels_like_c   0.988         1.000
True
            registered    cnt
registered       1.000  0.972
cnt              0.972  1.000


**Напишіть словами:** Які є альтернативні пояснення кореляції між температурою й попитом (наприклад, сезон)?

**Відповідь:** Кореляція `cnt` з температурою 0.405, з вологістю -0.323. Між `temp_c` і `feels_like_c` кореляція 0.988, вони майже однакові. Інші пояснення зв'язку температури й попиту: сезон (влітку тепло й багато відпочивальників), година доби (вдень тепліше й людей більше) і рік (2012 мав більший попит). `cnt` і `registered` мають кореляцію 0.972 не тому, що це відкриття, а тому що `cnt = casual + registered`, це просто сума.

## 19. Квартилі, перцентилі та пошук викидів

**Навіщо це потрібно?** Знайти рекордну годину легко; складніше визначити, що означає «незвично висока». Перцентилі дозволяють зробити це прозоро.

`quantile(0.25)` — перший квартиль Q1; `quantile(0.75)` — третій квартиль Q3; `IQR = Q3 − Q1`. Поширене правило для пошуку кандидатів у викиди: значення нижче `Q1−1.5×IQR` або вище `Q3+1.5×IQR`. Це **сигнал для перевірки**, не наказ видалити рядок.

**Приклад — виконайте комірку та розгляньте результат:**

In [38]:
# Приклад: дивимося на квартилі та 95-й перцентиль.
df["cnt"].quantile([0.25, 0.5, 0.75, 0.95])

0.25     40.0
0.50    142.0
0.75    281.0
0.95    563.1
Name: cnt, dtype: float64

**✍️ Завдання 19 — тепер ви:**

1. Обчисліть Q1, Q3 та IQR для `cnt`.  
2. Порахуйте нижню й верхню межі за правилом 1.5×IQR.  
3. Скільки погодинних записів лежить **за межами**?  
4. Виведіть 5 найвищих таких записів разом із датою та годиною.

> **Підказка:** Для Q1: `df["cnt"].quantile(0.25)`; умови з `|` ставте в окремі дужки.

In [39]:
q1 = df["cnt"].quantile(0.25)
q3 = df["cnt"].quantile(0.75)
iqr = q3 - q1
lower = q1 - 1.5 * iqr
upper = q3 + 1.5 * iqr
outliers = df[(df["cnt"] < lower) | (df["cnt"] > upper)]
print(q1, q3, iqr)
print(lower, upper)
print(len(outliers))
print(outliers.nlargest(5, "cnt")[["dteday", "hr", "cnt"]])

40.0 281.0 241.0
-321.5 642.5
505
          dteday  hr  cnt
14773 2012-09-12  18  977
14964 2012-09-20  17  976
14748 2012-09-11  17  970
14725 2012-09-10  18  968
15084 2012-09-25  17  967


**Напишіть словами:** Чому пікова година в будній день може бути статистичним викидом, але не помилкою даних?

**Відповідь:** Пікова година в будній день, наприклад 17:00, має середнє 525.3 оренди, а межа викидів 642.5. Через те 505 рядків (2.9%) стали «викидами». Але це люди, які їдуть з роботи, тому значення справжні й нормальні для цього часу. Правило IQR лише показує, що їх варто перевірити, а не що це помилка.

## 20. Дати та щоденна агрегація

**Навіщо це потрібно?** Погодинні записи зручні для аналізу години пік, але питання «який день був найуспішнішим?» потребує зведення до рівня дня.

`groupby("dteday")["cnt"].sum()` обчислює всі оренди **за спостережені години одного дня**. Паралельно корисно рахувати, скільки годин є в кожному дні: відсутні години роблять порівняння денних сум менш чесним.

**Приклад — виконайте комірку та розгляньте результат:**

In [40]:
# Приклад: кількість оренд та спостережених годин за день.
daily = (
    df.groupby("dteday")
      .agg(total_rentals=("cnt", "sum"), observed_hours=("hr", "size"))
      .sort_index()
)
daily.head()

,total_rentals,observed_hours
dteday,,
2011-01-01,985,24
2011-01-02,801,23
2011-01-03,1349,22
2011-01-04,1562,23
2011-01-05,1600,23


**✍️ Завдання 20 — тепер ви:**

1. Визначте **5 днів із найбільшою сумою оренд**.  
2. Скільки днів мають **менше 24 погодинних записів**?  
3. Обчисліть для кожного дня **середню кількість оренд за спостережену годину**.  
4. Чи однакові лідери за денною сумою і за середнім на годину?

> **Підказка:** Використовуйте таблицю `daily`, створену в прикладі; нову метрику додайте як колонку.

In [41]:
print(daily.nlargest(5, "total_rentals"))
print((daily["observed_hours"] < 24).sum())
daily["avg_per_observed_hour"] = daily["total_rentals"] / daily["observed_hours"]
print(daily.nlargest(5, "avg_per_observed_hour").round(1))

            total_rentals  observed_hours
dteday                                   
2012-09-15           8714              24
2012-09-29           8555              24
2012-09-22           8395              24
2012-03-23           8362              24
2012-05-19           8294              24
76
            total_rentals  observed_hours  avg_per_observed_hour
dteday                                                          
2012-09-15           8714              24                  363.1
2012-09-29           8555              24                  356.5
2012-09-22           8395              24                  349.8
2012-03-23           8362              24                  348.4
2012-05-19           8294              24                  345.6


**Напишіть словами:** Чому день із 20 спостереженими годинами може виглядати гірше за день із 24 годинами?

**Відповідь:** Сума оренд залежить від кількості записів за день. 76 днів мають менше 24 записів, тому їх сума менша не через слабкий попит, а через відсутні години. Топ-5 днів за сумою (15.09.2012 — 8714, 29.09.2012, 22.09.2012, 23.03.2012, 19.05.2012) і за середнім на годину ті самі, бо в усіх цих днів є всі 24 години.

## 21. Ковзне середнє: `rolling()`

**Навіщо це потрібно?** Щоденний попит шумний: на нього впливають конкретна погода, день тижня та випадкові події. Ковзне середнє згладжує коливання.

`rolling(7).mean()` обчислює середнє за поточним і попередніми 6 **рядками**. Після добової агрегації це зазвичай 7 дат, але треба перевірити, чи дат немає пропущених. `min_periods=7` не дозволяє рахувати неповне 7-денне вікно.

**Приклад — виконайте комірку та розгляньте результат:**

In [42]:
# Приклад: щоденна сума і ковзне середнє за 7 наявних дат.
daily = daily.sort_index().copy()
daily["rolling_7"] = daily["total_rentals"].rolling(7, min_periods=7).mean()
daily[["total_rentals", "rolling_7"]].head(12).round(1)

,total_rentals,rolling_7
dteday,,
2011-01-01,985,NaN
2011-01-02,801,NaN
2011-01-03,1349,NaN
2011-01-04,1562,NaN
2011-01-05,1600,NaN
2011-01-06,1606,NaN
2011-01-07,1510,1344.7
2011-01-08,959,1341.0
2011-01-09,822,1344.0


**✍️ Завдання 21 — тепер ви:**

1. Створіть **14-денне ковзне середнє** для `total_rentals`.  
2. Знайдіть дату максимального значення цього середнього.  
3. Порівняйте його зі звичайним максимумом денних оренд.  
4. Поясніть, чому перші 13 значень 14-денного вікна можуть бути порожніми.

> **Підказка:** Спробуйте `daily["total_rentals"].rolling(14, min_periods=14).mean()`.

In [43]:
daily["rolling_14"] = daily["total_rentals"].rolling(14, min_periods=14).mean()
print(daily["rolling_14"].idxmax(), round(daily["rolling_14"].max(), 1))
print(daily["total_rentals"].idxmax(), daily["total_rentals"].max())
print(daily["rolling_14"].isna().sum())

2012-09-22 00:00:00 7576.0
2012-09-15 00:00:00 8714
13


**Напишіть словами:** Чим короткий сплеск відрізняється від тривалого підвищення попиту?

**Відповідь:** Короткий сплеск це один-два дні з великим попитом, наприклад 15.09.2012 мав 8714 оренд. Тривале підвищення це багато днів підряд, і його видно у ковзному середньому. Максимум 14-денного середнього 7576.0 був 22.09.2012, а звичайний максимум 8714. Перші 13 значень порожні (їх рівно 13), бо для 14-денного вікна потрібно 14 днів, а на початку їх ще немає.

---
# Частина 5. Додаткові виклики (за бажанням)

Ці задачі ближчі до реальної аналітичної роботи: порівнюємо спостереження з типовим значенням **для схожих умов**, а не зі всім датасетом одразу.

## 22. `transform()`: порівняння кожної години з типовою

**Навіщо це потрібно?** Годину о 8:00 недоцільно порівнювати із середнім за всі 24 години: будні й ночі мають різний режим попиту.

На відміну від `groupby().mean()`, який повертає одну відповідь на групу, `transform("mean")` повертає **значення для кожного початкового рядка**: середнє його групи. Так можна створити базову лінію та відхилення.

**Приклад — виконайте комірку та розгляньте результат:**

In [44]:
# Приклад: порівнюємо кожний рядок із середнім для цієї ж години доби.
df["typical_for_hour"] = df.groupby("hr")["cnt"].transform("mean")
df["delta_from_hour"] = df["cnt"] - df["typical_for_hour"]
df[["dteday", "hr", "cnt", "typical_for_hour", "delta_from_hour"]].head()

,dteday,hr,cnt,typical_for_hour,delta_from_hour
0,2011-01-01,0,16,53.898072,-37.898072
1,2011-01-01,1,40,33.375691,6.624309
2,2011-01-01,2,32,22.869930,9.130070
3,2011-01-01,3,13,11.727403,1.272597
4,2011-01-01,4,1,6.352941,-5.352941


**✍️ Завдання 22 — тепер ви:**

1. Створіть базову метрику для **комбінації `workingday + hr`**.  
2. Обчисліть відхилення кожного запису від середнього своєї групи.  
3. Знайдіть 5 найбільш **негативних відхилень**.  
4. Припустіть, чому вони могли виникнути (це лише гіпотези).

> **Підказка:** Перенесіть приклад, замінивши одну групувальну колонку на список із двох.

In [45]:
df["baseline"] = df.groupby(["workingday", "hr"])["cnt"].transform("mean")
df["delta"] = df["cnt"] - df["baseline"]
print(df.nsmallest(5, "delta")[["dteday", "hr", "workingday", "cnt", "baseline", "delta"]])

          dteday  hr  workingday  cnt    baseline       delta
586   2011-01-26  17           1   26  525.290581 -499.290581
17205 2012-12-24  17           1   26  525.290581 -499.290581
8065  2011-12-07  17           1   52  525.290581 -473.290581
17252 2012-12-26  17           1   53  525.290581 -472.290581
17206 2012-12-24  18           1   26  492.226908 -466.226908


**Напишіть словами:** Чому порівняння з нормою «та сама година + той самий тип дня» точніше?

**Відповідь:** У будній і вихідний день ті самі години поводяться по-різному, наприклад о 17:00 у будній день середнє 525.3, а у вихідний 323.5. Тому порівнювати треба з тією ж годиною і тим же типом дня. Найбільші мінуси: 26.01.2011 о 17:00 (26 замість 525.3, різниця -499.3), 24.12.2012 о 17:00 (-499.3), 07.12.2011 о 17:00 (-473.3), 26.12.2012 о 17:00 (-472.3), 24.12.2012 о 18:00 (-466.2). Гіпотези: у всіх цих рядках погода була 3 (опади), а вологість 93–100%, до того ж 24 і 26 грудня це святковий період.

## 23. Перевірка гіпотези з `assign()` та `query()`

**Навіщо це потрібно?** Дослідник формулює твердження, створює необхідний показник і перевіряє його на підмножині даних.

`assign()` створює копію таблиці з додатковою колонкою, а `query()` дозволяє писати прості умови текстом. Це зручно для невеликих відтворюваних перевірок. Обережно: частка може бути не визначена, якщо знаменник дорівнює нулю.

**Приклад — виконайте комірку та розгляньте результат:**

In [46]:
# Приклад: скільки записів з cnt > 0 у певному часовому проміжку.
active_evenings = (
    df.assign(is_evening=df["hr"].between(18, 21))
      .query("is_evening and cnt > 0")
)
print("Вечірніх рядків з ненульовими орендами:", len(active_evenings))

Вечірніх рядків з ненульовими орендами: 2912


**✍️ Завдання 23 — тепер ви:**

1. Знайдіть усі рядки, де **`temp_c > 25`** та **`workingday == 0`**.  
2. Порахуйте середнє `cnt` у цій групі.  
3. Порівняйте із **неробочими годинами, де `temp_c <= 25`**.  
4. Чи достатньо такого порівняння, щоб зробити висновок про температуру?

> **Підказка:** Показуйте не лише середнє, а й `len(...)`, щоб бачити обсяг порівнюваних груп.

In [47]:
hot_off = df.query("temp_c > 25 and workingday == 0")
cool_off = df.query("temp_c <= 25 and workingday == 0")
print(len(hot_off), round(hot_off["cnt"].mean(), 1))
print(len(cool_off), round(cool_off["cnt"].mean(), 1))

801 343.3
4713 153.9


**Напишіть словами:** Назвіть хоча б два інші фактори, які могли впливати на побачену різницю.

**Відповідь:** Теплих неробочих годин (понад 25 °C) було 801, середнє 343.3 оренди за годину. Інших неробочих годин (25 °C і менше) було 4713, середнє 153.9. Різниця 189.4, але цього мало для висновку про температуру. Інші фактори: сезон (спека буває влітку, коли люди у відпустці), година доби (вдень тепліше, і їздять більше), рік (у 2012 попит був більший), опади й вітер.

---
# Частина 6. Мініпроєкт: поясніть попит

Тепер спробуйте мислити як аналітик. Для кожної гіпотези потрібні: **що саме перевіряємо → як обчислюємо → які числа отримали → що вони означають → які є обмеження**.

## 24. Ваші три гіпотези (обов’язково)

Обирайте **три різні** твердження. Можете скористатись варіантами нижче **або вигадати свої**:

- **Пікові години** зареєстрованих користувачів відрізняються в робочі й неробочі дні.
- **Зв’язок температури й попиту** відрізняється за сезонами.
- У різні **погодні категорії** різниця між випадковими та зареєстрованими користувачами неоднакова.
- З 2011 до 2012 року середній погодинний попит виріс **не однаково в усі місяці**.
- У найхолодніші години спостерігається нижчий попит, але різниця залежить від часу доби.

**Для кожної гіпотези:** створіть окрему таблицю Pandas, запишіть числовий результат, сформулюйте висновок. Якщо гіпотеза не підтвердилася — це теж правильний результат дослідження.

In [48]:
print("Гіпотеза 1: пікові години зареєстрованих різні в робочі й неробочі дні")
reg_by_hour = df.groupby(["workingday", "hr"])["registered"].mean().round(1)
print(reg_by_hour.groupby(level=0).nlargest(3).droplevel(0))

print("Гіпотеза 2: зв'язок температури й попиту різний у різні сезони")
corr_by_season = pd.Series(
    {s: g["temp_c"].corr(g["cnt"]) for s, g in df.groupby("season")}
).round(3)
print(corr_by_season)
print(df.groupby("season")["temp_c"].mean().round(1))

print("Гіпотеза 3: ріст попиту з 2011 до 2012 різний у різні місяці")
month_year = df.pivot_table(index="mnth", columns="yr", values="cnt", aggfunc="mean")
growth = ((month_year[1] / month_year[0] - 1) * 100).round(1)
print(month_year.round(1))
print(growth)
print(growth.idxmax(), growth.max())
print(growth.idxmin(), growth.min())

Гіпотеза 1: пікові години зареєстрованих різні в робочі й неробочі дні


workingday  hr
0           12    240.0
            13    236.3
            14    225.1
1           17    468.4
            8     454.7
            18    441.8
Name: registered, dtype: float64
Гіпотеза 2: зв'язок температури й попиту різний у різні сезони
1    0.446
2    0.408
3    0.345
4    0.342
dtype: float64
season
1     6.1
2    17.6
3    25.2
4    11.9
Name: temp_c, dtype: float64
Гіпотеза 3: ріст попиту з 2011 до 2012 різний у різні місяці
yr        0      1
mnth              
1      55.5  130.6
2      74.3  149.0
3      87.7  221.9
4     131.9  242.7
5     182.6  263.3
6     199.3  281.7
7     190.0  273.7
8     187.0  288.3
9     177.7  303.6
10    166.2  280.8
11    142.1  212.6
12    117.8  166.7
mnth
1     135.2
2     100.6
3     152.9
4      83.9
5      44.2
6      41.3
7      44.1
8      54.2
9      70.8
10     68.9
11     49.6
12     41.5
dtype: float64
3 152.9
6 41.3


| № | Ваша гіпотеза | Які групи / метрики порівнювали | Результат із числами | Підтвердилася? | Що може спотворювати висновок? |
|---|---|---|---|---|---|
| 1 | Пікові години зареєстрованих різні в робочі й неробочі дні | Середні `registered` за годиною, окремо для `workingday` = 1 і 0 | У робочі дні пік о 17:00 (468.4), також о 8:00 (454.7). У неробочі пік о 12:00 (240.0) | Так | Це середні значення, ми не знаємо, куди саме їдуть люди. Робочих годин 11865, неробочих 5514 |
| 2 | Зв'язок температури й попиту різний за сезонами | Кореляція `temp_c` і `cnt` у кожному сезоні | Зима 0.446, весна 0.408, літо 0.345, осінь 0.342 | Частково: у всіх сезонах зв'язок позитивний, різниця невелика (до 0.10) | Кореляція не доводить причину. Впливають година доби, погода й рік |
| 3 | З 2011 до 2012 року попит виріс не однаково в усіх місяцях | Середній `cnt` за годину по місяцях у 2011 і 2012, ріст у % | Березень +152.9% (87.7 → 221.9), січень +135.2%, червень лише +41.3% (199.3 → 281.7) | Так | У 2011 взимку початкові значення малі, тому відсотки великі. Є лише 2 роки даних |

## 25. Підсумковий звіт (обов’язково)

Напишіть 8–12 речень **своїми словами**, користуючись числами зі своїх розрахунків.

**1. Що ви дізналися про попит?** (час, день, сезон, погода)  
У будній день попит найбільший о 17:00 (461.5 оренди за годину в середньому), о 18:00 (425.5) і о 8:00 (359.0), а вночі о 4:00 лише 6.4. За сезонами літо найкраще (236.0), зима найгірша (111.1). У ясну погоду в середньому 204.9 оренди за годину, у дощ чи сніг 111.6. У 2012 році попит був більший, ніж у 2011 (234.7 проти 143.8).

**2. Як поводяться `casual` та `registered`?**  
Зареєстрованих оренд 2672662 (81.2%), незареєстрованих 620017 (18.8%). Зареєстровані їздять у піки о 8:00 і 17:00 у робочі дні. Незареєстровані частіше їздять вдень, їх пік о 14:00 (75.6 за годину), а у вихідні вони мають 139.6.

**3. Які три практичні рішення ви запропонували б компанії?** (приклад напряму: коли збільшувати доступність велосипедів; не пишіть просто «покращити сервіс»)  
Рекомендація 1: Перед піками о 8:00 і 17:00–18:00 у робочі дні перевіряти, щоб на станціях було достатньо велосипедів, бо тоді попит 359–461 за годину.  
Рекомендація 2: Ремонт і обслуговування велосипедів робити вночі з 2 до 4 години, бо тоді попит лише 6–23 оренди за годину.  
Рекомендація 3: Влітку й на початку осені (червень–вересень) тримати більше велосипедів, а взимку менше (111.1 проти 236.0 за годину), і вихідного дня вдень пропонувати знижки для незареєстрованих.

**4. Які обмеження цього дослідження?** Згадайте принаймні **дві**: історичні дані, пропущені години, розмір груп, змішані фактори, кореляція ≠ причинність, відсутність інформації про індивідуальні поїздки.  
Дані тільки за 2011–2012 роки, тому вони можуть не підходити до сьогоднішнього ринку. Бракує 165 годин, а категорія погоди 4 має лише 3 записи. Кореляція не доводить причину. Ми не маємо даних про окремі поїздки, тільки кількість оренд за годину.

## Критерії оцінювання — 100 балів

| Розділ | Бали |
|---|---:|
| Частина 1: базові операції та пояснення (1–5) | 20 |
| Частина 2: фільтрація, якість, перетворення (6–11) | 25 |
| Частина 3: групування та порівняння (12–17) | 25 |
| Частина 4: кореляції, викиди, дати (18–21) | 15 |
| Три гіпотези з числовими доказами (24) | 10 |
| Підсумковий звіт і практичні висновки (25) | 5 |
| **Разом** | **100** |

**Бонус:** за кожне якісно виконане додаткове завдання 22 або 23 — **+5 балів** (максимум +10).

**Оцінюється не «красива відповідь», а логіка аналізу.** Навіть якщо результат виявився неочікуваним, добре обґрунтований висновок — це плюс.

## Шпаргалка: найчастіші операції Pandas

| Хочу зробити | Шаблон |
|---|---|
| Подивитись початок | `df.head(5)` |
| Дізнатись розмір | `df.shape` |
| Побачити типи | `df.info()` |
| Знайти статистики | `df["cnt"].describe()` |
| Порахувати категорії | `df["season"].value_counts()` |
| Вибрати кілька колонок | `df[["hr", "cnt"]]` |
| Відібрати рядки | `df[df["hr"] == 8]` |
| Поєднати умови | `df[(умова1) & (умова2)]` |
| Сортувати | `df.sort_values("cnt", ascending=False)` |
| Знайти пропуски | `df.isna().sum()` |
| Створити колонку | `df["new"] = ...` |
| Порівняти групи | `df.groupby("hr")["cnt"].mean()` |
| Декілька статистик | `df.groupby("hr").agg(...)` |
| Матриця середніх | `df.pivot_table(...)` |
| Частоти перетинів | `pd.crosstab(...)` |
| Кореляція | `df[["temp", "cnt"]].corr()` |
| Квартиль | `df["cnt"].quantile(0.75)` |
| Ковзне середнє | `daily["total_rentals"].rolling(7).mean()` |

### Перед здачею

- [ ] `hour.csv` покладено поруч з ноутбуком (або завантажено вручну в Colab).
- [ ] Усі обов’язкові завдання 1–21 виконані **кодом**, а не лише текстом.
- [ ] Після кожного завдання є коротке пояснення отриманих результатів.
- [ ] Є **три гіпотези**, їх перевірки та висновки.
- [ ] Є **три конкретні рекомендації** компанії.
- [ ] Ноутбук виконується від початку до кінця без помилок, якщо `hour.csv` лежить у папці проєкту.

**Не потрібно:** писати нейромережі, навчати моделі, встановлювати бібліотеки для графіків, шукати готові відповіді в інтернеті.